# Ames Housing Price Prediction — Empirical Evaluation

Self-contained notebook for Google Colab and local execution. Trains OLS Linear Regression and Random Forest on the Ames Housing Dataset (`2,930` rows, `81` columns), computes holdout metrics, and plots correlations, MDI feature importances, and residuals.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    r2_score,
    root_mean_squared_error,
)
from sklearn.model_selection import train_test_split

LOCAL_PATH = "../data/ames_housing.csv"
RAW_URL = "https://raw.githubusercontent.com/GaneshArwan/housing_price_prediction/main/data/ames_housing.csv"

df = pd.read_csv(LOCAL_PATH if os.path.exists(LOCAL_PATH) else RAW_URL)
print(f"Raw shape: {df.shape}")
print("Columns with missing values:", df.isna().sum()[df.isna().sum() > 0].to_dict())
df.head(3)

In [ ]:
processed = df.copy()
cols_to_drop = ["Id", "PID", "Alley", "PoolQC", "Fence", "MiscFeature"]
processed.drop(columns=[c for c in cols_to_drop if c in processed.columns], inplace=True)
if "Sale_Price" in processed.columns:
    processed.rename(columns={"Sale_Price": "SalePrice"}, inplace=True)

num_cols = processed.select_dtypes(include=["number"]).columns
processed[num_cols] = processed[num_cols].fillna(processed[num_cols].median())
cat_cols = processed.select_dtypes(exclude=["number"]).columns
processed[cat_cols] = processed[cat_cols].fillna("Unknown")
processed = pd.get_dummies(processed, columns=cat_cols, drop_first=True)

X = processed.drop(columns=["SalePrice"])
y = processed["SalePrice"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

lr = LinearRegression().fit(X_train, y_train)
rf = RandomForestRegressor(n_estimators=100, max_depth=None, random_state=42).fit(X_train, y_train)

def summarize(name, model):
    preds = model.predict(X_test)
    std = y_test < 400000
    lux = y_test >= 400000
    return {
        "Model": name,
        "RMSE ($)": round(root_mean_squared_error(y_test, preds), 2),
        "MAE ($)": round(mean_absolute_error(y_test, preds), 2),
        "R2": round(r2_score(y_test, preds), 4),
        "MAPE (%)": round(mean_absolute_percentage_error(y_test, preds) * 100, 2),
        "Std RMSE (<$400k)": round(root_mean_squared_error(y_test[std], preds[std]), 2),
        "Lux RMSE (>=$400k)": round(root_mean_squared_error(y_test[lux], preds[lux]), 2),
    }

pd.DataFrame([summarize("Linear Regression", lr), summarize("Random Forest", rf)])

In [ ]:
numeric_df = df.select_dtypes(include=["number"])
top_corr = numeric_df.corr()["Sale_Price"].sort_values(ascending=False).index[:11]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.heatmap(numeric_df[top_corr].corr(), annot=True, fmt=".2f", cmap="coolwarm", ax=axes[0])
axes[0].set_title("Top 10 Numeric Correlations")

imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).head(15)
sns.barplot(x=imp.values, y=imp.index, ax=axes[1])
axes[1].set_title("Top 15 RF MDI Feature Importances")

rf_preds = rf.predict(X_test)
sns.scatterplot(x=rf_preds, y=y_test - rf_preds, alpha=0.5, ax=axes[2])
axes[2].axhline(0, color="red", linestyle="--")
axes[2].set_title("Random Forest Residuals")
axes[2].set_xlabel("Predicted ($)")
axes[2].set_ylabel("Residual ($)")
plt.tight_layout()
plt.show()

## Key Findings & Next Steps
1. **Ordinal one-hot dilution:** `Overall_Qual` is stored as text categories and split into 9 one-hot dummies (`2.20%` combined MDI), leaving `Garage_Cars` (`29.78%`), `Gr_Liv_Area` (`14.95%`), and `Year_Built` (`10.89%`) as the top single-column MDI splits.
2. **Luxury tail error:** On `n=566` standard homes (`<$400k`), Random Forest RMSE is `$24,242.60`, rising `3.48x` to `$84,378.68` on `n=20` luxury homes (`>=$400k`).
3. **Recommended upgrades:** Ordinal-encode quality ratings, train on `log1p(SalePrice)`, and benchmark gradient boosting with 5-fold cross-validation.